# Clase 171 — Aceleración con GPU

Configurar y aprovechar la GPU para deep learning: verificar dispositivos, controlar la memoria,
activar **mixed precision** (float16/bfloat16, que duplica el throughput en GPUs modernas) y
profilear para encontrar cuellos de botella.

Requiere: `tensorflow` (opcional). El código es correcto; sin GPU/TF no se ejecuta aquí.

## 1. Verificar la GPU disponible

In [ ]:
try:
    import tensorflow as tf
    TF_OK = True
    print("tensorflow", tf.__version__)
except Exception:
    TF_OK = False
    print("tensorflow no instalado -> se muestra la API (no se ejecuta)")

if TF_OK:
    gpus = tf.config.list_physical_devices("GPU")
    print("GPUs fisicas:", gpus)
    print("GPUs logicas:", tf.config.list_logical_devices("GPU"))
    if not gpus:
        print("(sin GPU: TF corre en CPU; el codigo de abajo es la configuracion correcta)")
else:
    print("tf.config.list_physical_devices('GPU')  /  torch.cuda.is_available()")

## 2. Memory growth: no reservar toda la VRAM al inicio

Por defecto TF reserva casi toda la memoria de la GPU. `set_memory_growth(True)` hace que la
reserva crezca bajo demanda (útil para compartir la GPU).

In [ ]:
if TF_OK:
    for gpu in tf.config.list_physical_devices("GPU"):
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
            print("memory growth activado en", gpu.name)
        except RuntimeError as e:
            print("debe configurarse antes de inicializar la GPU:", e)
else:
    print("tf.config.experimental.set_memory_growth(gpu, True)")

## 3. Mixed precision

- `mixed_float16` (Volta+): requiere **loss scaling** (automático con Keras) para evitar underflow.
- `mixed_bfloat16` (Ampere+/TPU): mismo rango que float32, **sin** loss scaling.

Los pesos "master" quedan en float32; forward/backward usan 16 bits → 1.5-3× más rápido.

In [ ]:
if TF_OK:
    from tensorflow import keras
    keras.mixed_precision.set_global_policy("mixed_float16")     # 'mixed_bfloat16' en Ampere+
    print("policy global:", keras.mixed_precision.global_policy().name)

    model = keras.Sequential([
        keras.Input(shape=(784,)),
        keras.layers.Dense(256, activation="relu"),
        keras.layers.Dense(10, dtype="float32"),                # ultima capa en float32 (estabilidad)
    ])
    print("ultima capa dtype:", model.layers[-1].dtype)         # float32 para softmax/loss estable
else:
    print("keras.mixed_precision.set_global_policy('mixed_float16')")

## 4. Colocación explícita con `tf.device`

In [ ]:
if TF_OK:
    dev = "/GPU:0" if tf.config.list_physical_devices("GPU") else "/CPU:0"
    with tf.device(dev):
        a = tf.random.normal((512, 512))
        b = tf.matmul(a, a)
    print(f"matmul ejecutado en {dev}; resultado shape {b.shape}")
else:
    print("with tf.device('/GPU:0'): ...   # coloca las ops en la GPU 0")

## 5. Profiling y cuellos de botella

```python
tb = keras.callbacks.TensorBoard(log_dir="logs/", profile_batch=(5, 10))
model.fit(ds, epochs=1, callbacks=[tb])   # abrir la pestaña Profiler en TensorBoard
```

Regla de lectura: si la GPU está al ~30% y la CPU alta, el cuello de botella es el **data
loading** → optimizar el pipeline `tf.data` (`prefetch`, `cache`, `num_parallel_calls`).
GPUs típicas 2026: H100/H200 (server), RTX 5090 (consumer).

## 6. Micro-benchmark de matmul (ejecutable)

Medimos un matmul grande con numpy como referencia de CPU. En GPU con mixed precision el mismo
cómputo sería 1.5-3× más rápido; aquí solo ilustramos cómo cronometrar.

In [ ]:
import numpy as np, time
np.random.seed(0)
A = np.random.rand(1024, 1024).astype("float32")
t0 = time.perf_counter()
for _ in range(5):
    B = A @ A
ms = (time.perf_counter() - t0) / 5 * 1000
print(f"matmul 1024x1024 (numpy CPU): {ms:.2f} ms/run")
print("en GPU + mixed_float16 el mismo matmul rinde 1.5-3x mas (segun arquitectura)")

## Ejercicios

1. Imprimir `list_physical_devices('GPU')`, `list_logical_devices('GPU')` y `nvidia-smi`.
2. Activar `set_memory_growth(True)` y verificar el consumo con `nvidia-smi` durante el training.
3. Re-entrenar un modelo con `mixed_float16` y medir el speedup wall-clock (esperado ≥ 1.5×).
4. Profilear con TensorBoard, identificar el bottleneck y aplicar más `prefetch`.

## Conclusiones

- `tf.config.list_physical_devices('GPU')` confirma que TF ve la GPU.
- `set_memory_growth(True)` evita reservar toda la VRAM al arrancar.
- Mixed precision (`mixed_float16`/`mixed_bfloat16`) acelera 1.5-3× en GPUs modernas.
- El profiler revela si el cuello está en compute o en data loading; casi siempre es el pipeline.

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y comentadas de los ejercicios del README. Las de **núcleo numérico** son ejecutables (con `assert` de verificación); las de frameworks/servicios no instalados aquí (TF, PyTorch, diffusers, Gymnasium, GCP…) se muestran como **código real de referencia** listo para copiar en un entorno con esas dependencias.

### Ejercicio 1 — Verificar GPUs

```python
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))
print(tf.config.list_logical_devices('GPU'))
```
```bash
nvidia-smi   # uso, memoria, procesos, version de driver/CUDA
```

### Ejercicio 2 — Memory growth

```python
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)  # no reservar toda la VRAM al inicio
```

### Ejercicio 3 — Mixed precision (float16, núcleo ejecutable)

`mixed_float16` usa float16 en el cómputo (más rápido en tensor cores) manteniendo float32 en los pesos maestros. Ilustramos con NumPy el **trade-off numérico** de float16.

In [ ]:
import numpy as np
x = np.array([1.0, 1e-4, 65504.0, 70000.0], dtype=np.float32)
x16 = x.astype(np.float16)

# float16: rango ~ +-65504; 70000 desborda a inf
assert np.isinf(x16[3]), 'float16 desborda por encima de 65504 (de ahi el loss scaling)'
# Precision reducida: error relativo mayor que float32
rel = abs(float(x16[1]) - x[1]) / x[1]
assert rel > 0, 'float16 introduce error de redondeo'
print('float16 max finito=65504; 70000 ->', x16[3], '| err rel 1e-4:', round(rel, 4))
print('OK: por eso mixed precision usa loss scaling y pesos maestros en float32')

Aplicación real en Keras:

```python
from tensorflow import keras
keras.mixed_precision.set_global_policy('mixed_float16')
# La ultima capa conviene dejarla en float32: Dense(10, dtype='float32')
# Speedup tipico: 1.5-2x en V100, 2-3x en A100/H100.
```

### Ejercicio 4 — Profiling con TensorBoard

```python
cb = keras.callbacks.TensorBoard(log_dir='logs', profile_batch=(5, 10))
model.fit(..., callbacks=[cb])   # abrir pestana Profiler en TensorBoard
```

### Ejercicio 5 — Diagnóstico de bottleneck

```python
# GPU al 30% de uso => el cuello de botella es el data pipeline, no el computo.
ds = (ds.cache()
        .shuffle(10_000)
        .batch(256)
        .prefetch(tf.data.AUTOTUNE))   # solapa carga de datos con computo en GPU
```